# Structured Data Exploitation Zone

This notebook transforms the cleaned Trusted Zone tables in ClickHouse into exploitation-ready dimensional models and analytical marts.

The design follows two rules:

- `natural_disaster_tweets`: text-oriented feature extraction.
- `global_warming_dataset`, `temperature_change`, and `co2_emission_by_vehicles`: dimensional modelling plus denormalized analytical marts.

## 1. Environment Setup

In [1]:
# Import the ClickHouse client used to create exploitation-zone tables.
from string import Formatter

import clickhouse_connect
import pandas as pd

# Source database: cleaned and standardized Trusted Zone tables.
TRUSTED_DB = "bi_analytics"

# Target database: curated tables for analytics, dashboards, and ML features.
EXPLOITATION_DB = "exploitation_analytics"

# Logical source table names used throughout the exploitation model.
GW_TABLE = "global_warming_dataset"
TEMP_TABLE = "temperature_change"
VEHICLE_TABLE = "co2_emission_by_vehicles"
TWEET_TABLE = "natural_disaster_tweets"

TRUSTED_TABLES = [TWEET_TABLE, GW_TABLE, TEMP_TABLE, VEHICLE_TABLE]

# Candidate lists keep exploitation resilient to small trusted-zone naming improvements.
# The first item is the preferred canonical lowercase name; later items are accepted legacy forms.
COLUMN_CANDIDATES = {
    GW_TABLE: {},
    TEMP_TABLE: {},
    TWEET_TABLE: {},
    VEHICLE_TABLE: {
        "engine_size_l": ["engine_size_l", "engine_sizel"],
        "co2_emissions_g_km": ["co2_emissions_g_km", "co2_emissionsg_km"],
    },
}

# Create one reusable ClickHouse connection for the whole notebook.
client = clickhouse_connect.get_client(
    host="clickhouse",
    port=8123,
    username="analytics",
    password="analytics_secret",
)


def quote_identifier(identifier: str) -> str:
    """Quote a ClickHouse identifier without changing its case."""
    return "`" + identifier.replace("`", "``") + "`"


def table_ref(database_name: str, table_name: str) -> str:
    """Build a fully-qualified ClickHouse table reference."""
    return f"{quote_identifier(database_name)}.{quote_identifier(table_name)}"


def trusted_ref(table_name: str) -> str:
    """Return a fully-qualified Trusted Zone table reference."""
    return table_ref(TRUSTED_DB, table_name)


def exploitation_ref(table_name: str) -> str:
    """Return a fully-qualified Exploitation Zone table reference."""
    return table_ref(EXPLOITATION_DB, table_name)


_TRUSTED_COLUMN_CACHE: dict[str, set[str]] = {}


def trusted_columns(table_name: str) -> set[str]:
    """Read the actual Trusted Zone schema from ClickHouse once per table."""
    if table_name not in _TRUSTED_COLUMN_CACHE:
        rows = client.query(
            f"""
            SELECT name
            FROM system.columns
            WHERE database = '{TRUSTED_DB}'
              AND table = '{table_name}'
            """
        ).result_rows
        _TRUSTED_COLUMN_CACHE[table_name] = {row[0] for row in rows}
    return _TRUSTED_COLUMN_CACHE[table_name]


def resolve_column(table_name: str, logical_name: str) -> str:
    """Resolve a logical business field to the physical Trusted Zone column name."""
    columns = trusted_columns(table_name)
    candidates = COLUMN_CANDIDATES.get(table_name, {}).get(logical_name, [logical_name])

    for candidate in candidates:
        if candidate in columns:
            return candidate

    raise KeyError(
        f"Cannot resolve column '{logical_name}' in {trusted_ref(table_name)}. "
        f"Tried {candidates}; available columns: {sorted(columns)}"
    )


def source_col(table_name: str, logical_name: str) -> str:
    """Return a quoted physical source column for generated SQL."""
    return quote_identifier(resolve_column(table_name, logical_name))


# Create the exploitation database if this notebook is executed for the first time.
client.command(f"CREATE DATABASE IF NOT EXISTS {quote_identifier(EXPLOITATION_DB)}")
print(f"Connected to ClickHouse. Target database: {EXPLOITATION_DB}")


Connected to ClickHouse. Target database: exploitation_analytics


## 2. Trusted Zone Validation

In [2]:
# Validate that every expected trusted table exists and contains rows before modelling.
for table_name in TRUSTED_TABLES:
    row_count = client.query(f"SELECT count() FROM {trusted_ref(table_name)}").first_row[0]
    columns = trusted_columns(table_name)
    print(f"{table_name:<32} {row_count:>10,} rows | {len(columns):>3} columns")


natural_disaster_tweets             127,527 rows |   5 columns
global_warming_dataset              100,000 rows |  26 columns
temperature_change                  241,893 rows |  14 columns
co2_emission_by_vehicles              5,988 rows |  12 columns


## 3. Helper Functions

In [3]:
# This helper makes every table creation idempotent: rerunning the notebook replaces old outputs.
def recreate_table(table_name: str, select_sql: str, order_by: str) -> None:
    """Create an exploitation table from a SELECT statement in an idempotent way."""
    full_name = exploitation_ref(table_name)
    client.command(f"DROP TABLE IF EXISTS {full_name}")
    client.command(
        f"""
        CREATE TABLE {full_name}
        ENGINE = MergeTree()
        ORDER BY {order_by}
        SETTINGS allow_nullable_key = 1
        AS
        {select_sql}
        """
    )
    row_count = client.query(f"SELECT count() FROM {full_name}").first_row[0]
    print(f"Created {full_name:<55} {row_count:>10,} rows")


def template_fields(template: str) -> set[str]:
    """Return placeholder names used by a SQL expression template."""
    return {field_name for _, field_name, _, _ in Formatter().parse(template) if field_name}


def render_source_template(table_name: str, template: str, column_map: dict[str, str]) -> str:
    """Replace logical placeholders with quoted Trusted Zone source columns."""
    resolved_columns = {
        placeholder: source_col(table_name, column_map.get(placeholder, placeholder))
        for placeholder in template_fields(template)
    }
    return template.format_map(resolved_columns)


def build_dimension_select(definition: dict, source_config: dict) -> str:
    """Build one SELECT DISTINCT branch for a reusable dimension definition."""
    table_name = source_config["table"]
    column_map = source_config.get("columns", {})
    fields = source_config.get("fields", definition["fields"])

    select_list = ",\n        ".join(
        f"{render_source_template(table_name, expression, column_map)} AS {quote_identifier(output_name)}"
        for output_name, expression in fields.items()
    )
    where_template = source_config.get("where", definition.get("where"))
    where_clause = ""
    if where_template:
        where_clause = f"\n    WHERE {render_source_template(table_name, where_template, column_map)}"

    return f"""
    SELECT DISTINCT
        {select_list}
    FROM {trusted_ref(table_name)}{where_clause}
    """.strip()


def recreate_dimension_table(table_name: str, definition: dict) -> None:
    """Create a dimension table by unioning configured distinct values from one or more sources."""
    select_sql = "\n    UNION DISTINCT\n".join(
        build_dimension_select(definition, source_config)
        for source_config in definition["sources"]
    )
    recreate_table(table_name, select_sql, definition["order_by"])


## 4. Dimension Tables

Dimension tables describe the main analytical entities used by the fact tables and marts. They make the model easier to query because repeated descriptive values are centralized into reusable lookup tables.

| Table | Grain | Purpose |
|---|---|---|
| `dim_year` | One row per year | Provides a shared time dimension and adds `decade` for long-term trend analysis. |
| `dim_country` | One row per country from the climate dataset | Connects country-level climate facts to readable country names. |
| `dim_area` | One row per temperature-change area | Describes geographic areas in the temperature dataset, including the M49 area code. |
| `dim_month` | One row per month label/code | Supports monthly temperature trend analysis. |
| `dim_disaster_type` | One row per disaster category | Standardizes tweet disaster categories such as flood, earthquake, or wildfire. |
| `dim_vehicle` | One row per vehicle configuration | Describes make, model, class, engine, transmission, and fuel attributes for vehicle-emission analysis. |

In [4]:
# Build reusable dimensions from compact definitions.
# Add a new source to a dimension by appending one item under "sources" instead of writing another SELECT block.
DIMENSION_DEFINITIONS = {
    "dim_year": {
        "order_by": "year_key",
        "fields": {
            "year_key": "toInt32({year})",
            "year": "toInt32({year})",
            "decade": "intDiv(toInt32({year}), 10) * 10",
        },
        "sources": [
            {"table": GW_TABLE},
            {"table": TEMP_TABLE},
        ],
    },
    "dim_country": {
        "order_by": "country_key",
        "fields": {
            "country_key": "cityHash64({country_name})",
            "country_name": "{country_name}",
        },
        "sources": [
            {"table": GW_TABLE, "columns": {"country_name": "country"}, "where": "{country_name} != ''"},
        ],
    },
    "dim_area": {
        "order_by": "area_key",
        "fields": {
            "area_key": "cityHash64({area_name})",
            "area_code_m49": "{area_code_m49}",
            "area_name": "{area_name}",
        },
        "sources": [
            {"table": TEMP_TABLE, "columns": {"area_name": "area"}, "where": "{area_name} != ''"},
        ],
    },
    "dim_month": {
        "order_by": "month_key",
        "fields": {
            "month_key": "{month_key}",
            "month_name": "{month_name}",
        },
        "sources": [
            {"table": TEMP_TABLE, "columns": {"month_key": "months_code", "month_name": "months"}, "where": "{month_name} != ''"},
        ],
    },
    "dim_disaster_type": {
        "order_by": "disaster_type_key",
        "fields": {
            "disaster_type_key": "cityHash64({disaster_type})",
            "disaster_type": "{disaster_type}",
        },
        "sources": [
            {"table": TWEET_TABLE, "where": "{disaster_type} != ''"},
        ],
    },
    "dim_vehicle": {
        "order_by": "vehicle_key",
        "fields": {
            "vehicle_key": "cityHash64({make}, {model}, {vehicle_class}, {transmission}, {fuel_type}, {engine_size_l}, {cylinders})",
            "make": "{make}",
            "model": "{model}",
            "vehicle_class": "{vehicle_class}",
            "engine_size_l": "{engine_size_l}",
            "cylinders": "{cylinders}",
            "transmission": "{transmission}",
            "fuel_type": "{fuel_type}",
        },
        "sources": [
            {"table": VEHICLE_TABLE},
        ],
    },
}

for dimension_name, definition in DIMENSION_DEFINITIONS.items():
    recreate_dimension_table(dimension_name, definition)


Created `exploitation_analytics`.`dim_year`                            124 rows
Created `exploitation_analytics`.`dim_country`                         195 rows
Created `exploitation_analytics`.`dim_area`                            247 rows
Created `exploitation_analytics`.`dim_month`                            17 rows
Created `exploitation_analytics`.`dim_disaster_type`                     5 rows
Created `exploitation_analytics`.`dim_vehicle`                       3,097 rows


## 5. Fact Tables

Fact tables store measurable events or observations at a clear grain. They keep the source data analytically structured while replacing repeated descriptive columns with dimension keys.

| Table | Grain | Purpose |
|---|---|---|
| `fact_climate_country_year` | One country per year | Stores climate, economy, population, emissions, policy, and environmental indicators for country-year analysis. |
| `fact_temperature_area_month` | One area per month per year | Stores monthly temperature-change values and quality flags for time-series trend analysis. |
| `fact_vehicle_emission` | One vehicle-emission record | Stores fuel-consumption and CO2-emission measurements linked to vehicle attributes. |

In [5]:
# Build fact tables at the natural analytical grain of each trusted structured dataset.
# These tables are normalized enough for reuse, while marts later denormalize them for BI consumption.
recreate_table(
    "fact_climate_country_year",
    f"""
    SELECT
        cityHash64({source_col(GW_TABLE, 'country')}) AS country_key,
        toInt32({source_col(GW_TABLE, 'year')}) AS year_key,
        {source_col(GW_TABLE, 'temperature_anomaly')} AS temperature_anomaly,
        {source_col(GW_TABLE, 'average_temperature')} AS average_temperature,
        {source_col(GW_TABLE, 'co2_emissions')} AS co2_emissions,
        {source_col(GW_TABLE, 'population')} AS population,
        {source_col(GW_TABLE, 'forest_area')} AS forest_area,
        {source_col(GW_TABLE, 'gdp')} AS gdp,
        {source_col(GW_TABLE, 'renewable_energy_usage')} AS renewable_energy_usage,
        {source_col(GW_TABLE, 'methane_emissions')} AS methane_emissions,
        {source_col(GW_TABLE, 'sea_level_rise')} AS sea_level_rise,
        {source_col(GW_TABLE, 'arctic_ice_extent')} AS arctic_ice_extent,
        {source_col(GW_TABLE, 'urbanization')} AS urbanization,
        {source_col(GW_TABLE, 'deforestation_rate')} AS deforestation_rate,
        {source_col(GW_TABLE, 'extreme_weather_events')} AS extreme_weather_events,
        {source_col(GW_TABLE, 'average_rainfall')} AS average_rainfall,
        {source_col(GW_TABLE, 'solar_energy_potential')} AS solar_energy_potential,
        {source_col(GW_TABLE, 'waste_management')} AS waste_management,
        {source_col(GW_TABLE, 'per_capita_emissions')} AS source_per_capita_emissions,
        {source_col(GW_TABLE, 'industrial_activity')} AS industrial_activity,
        {source_col(GW_TABLE, 'air_pollution_index')} AS air_pollution_index,
        {source_col(GW_TABLE, 'biodiversity_index')} AS biodiversity_index,
        {source_col(GW_TABLE, 'ocean_acidification')} AS ocean_acidification,
        {source_col(GW_TABLE, 'fossil_fuel_usage')} AS fossil_fuel_usage,
        {source_col(GW_TABLE, 'energy_consumption_per_capita')} AS energy_consumption_per_capita,
        {source_col(GW_TABLE, 'policy_score')} AS policy_score
    FROM {trusted_ref(GW_TABLE)}
    """,
    "(country_key, year_key)",
)

recreate_table(
    "fact_temperature_area_month",
    f"""
    SELECT
        cityHash64({source_col(TEMP_TABLE, 'area')}) AS area_key,
        toInt32({source_col(TEMP_TABLE, 'year')}) AS year_key,
        {source_col(TEMP_TABLE, 'months_code')} AS month_key,
        {source_col(TEMP_TABLE, 'domain')} AS domain,
        {source_col(TEMP_TABLE, 'element')} AS element,
        {source_col(TEMP_TABLE, 'unit')} AS unit,
        {source_col(TEMP_TABLE, 'value')} AS temperature_change_value,
        {source_col(TEMP_TABLE, 'flag')} AS flag,
        {source_col(TEMP_TABLE, 'flag_description')} AS flag_description
    FROM {trusted_ref(TEMP_TABLE)}
    """,
    "(area_key, year_key, month_key)",
)

recreate_table(
    "fact_vehicle_emission",
    f"""
    SELECT
        cityHash64(
            {source_col(VEHICLE_TABLE, 'make')},
            {source_col(VEHICLE_TABLE, 'model')},
            {source_col(VEHICLE_TABLE, 'vehicle_class')},
            {source_col(VEHICLE_TABLE, 'transmission')},
            {source_col(VEHICLE_TABLE, 'fuel_type')},
            {source_col(VEHICLE_TABLE, 'engine_size_l')},
            {source_col(VEHICLE_TABLE, 'cylinders')}
        ) AS vehicle_key,
        {source_col(VEHICLE_TABLE, 'fuel_consumption_city_l_100_km')} AS fuel_consumption_city_l_100_km,
        {source_col(VEHICLE_TABLE, 'fuel_consumption_hwy_l_100_km')} AS fuel_consumption_hwy_l_100_km,
        {source_col(VEHICLE_TABLE, 'fuel_consumption_comb_l_100_km')} AS fuel_consumption_comb_l_100_km,
        {source_col(VEHICLE_TABLE, 'fuel_consumption_comb_mpg')} AS fuel_consumption_comb_mpg,
        {source_col(VEHICLE_TABLE, 'co2_emissions_g_km')} AS co2_emissions_g_km
    FROM {trusted_ref(VEHICLE_TABLE)}
    """,
    "vehicle_key",
)


Created `exploitation_analytics`.`fact_climate_country_year`       100,000 rows
Created `exploitation_analytics`.`fact_temperature_area_month`     241,893 rows
Created `exploitation_analytics`.`fact_vehicle_emission`             5,988 rows


## 6. Tweet Feature Extraction

Tweets are different from the other structured datasets because their main analytical value is inside free text. This section converts tweet text into structured features and separates multi-valued hashtags into a bridge table.

| Table | Grain | Purpose |
|---|---|---|
| `fact_tweet_features` | One row per tweet | Converts tweet text into numerical and categorical features such as word count, hashtag count, URL count, alert-keyword flag, and simple sentiment label. |
| `bridge_tweet_hashtag` | One row per tweet-hashtag pair | Resolves the one-to-many relationship between tweets and hashtags, making hashtag frequency and hashtag-disaster analysis possible. |

In [6]:
# Extract lightweight text features directly in ClickHouse.
# ifNull(tweet_text, '') prevents Nullable(String) values from producing invalid Nullable(Array) results.
recreate_table(
    "fact_tweet_features",
    f"""
    SELECT
        {source_col(TWEET_TABLE, 'id')} AS tweet_id,
        cityHash64({source_col(TWEET_TABLE, 'disaster_type')}) AS disaster_type_key,
        {source_col(TWEET_TABLE, 'disaster_type')} AS disaster_type,
        length({source_col(TWEET_TABLE, 'emojis')}) AS emoji_count,
        ifNull({source_col(TWEET_TABLE, 'tweet_text')}, '') AS tweet_text,
        lengthUTF8(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, '')) AS text_length,
        length(splitByChar(' ', trim(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, '')))) AS word_count,
        countMatches(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), '#[A-Za-z0-9_]+') AS hashtag_count,
        countMatches(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), '@[A-Za-z0-9_]+') AS mention_count,
        countMatches(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'https?://|www\\.') AS url_count,
        countMatches(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), '!') AS exclamation_count,
        countMatches(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), '\\?') AS question_count,
        if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'youtube') > 0, 1, 0) AS contains_youtube,
        if(
            positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'emergency') > 0
            OR positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'warning') > 0
            OR positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'evacuation') > 0,
            1,
            0
        ) AS contains_alert_keyword,
        (
            if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'safe') > 0, 1, 0)
            + if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'rescue') > 0, 1, 0)
            + if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'help') > 0, 1, 0)
            - if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'damage') > 0, 1, 0)
            - if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'dead') > 0, 1, 0)
            - if(positionCaseInsensitive(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), 'destroyed') > 0, 1, 0)
        ) AS simple_sentiment_score,
        multiIf(simple_sentiment_score > 0, 'positive', simple_sentiment_score < 0, 'negative', 'neutral') AS simple_sentiment_label
    FROM {trusted_ref(TWEET_TABLE)}
    """,
    "tweet_id",
)


Created `exploitation_analytics`.`fact_tweet_features`             127,527 rows


In [7]:
# Split hashtags out of tweet text so each tweet-hashtag relationship becomes one analyzable row.
recreate_table(
    "bridge_tweet_hashtag",
    f"""
    SELECT
        {source_col(TWEET_TABLE, 'id')} AS tweet_id,
        lowerUTF8(replaceRegexpAll(hashtag, '^#', '')) AS hashtag
    FROM {trusted_ref(TWEET_TABLE)}
    ARRAY JOIN extractAll(ifNull({source_col(TWEET_TABLE, 'tweet_text')}, ''), '#[A-Za-z0-9_]+') AS hashtag
    WHERE hashtag != ''
    """,
    "(hashtag, tweet_id)",
)

# Split trusted emoji arrays out so each tweet-emoji relationship becomes one analyzable row.
recreate_table(
    "bridge_tweet_emoji",
    f"""
    SELECT
        {source_col(TWEET_TABLE, 'id')} AS tweet_id,
        emoji AS emoji
    FROM {trusted_ref(TWEET_TABLE)}
    ARRAY JOIN {source_col(TWEET_TABLE, 'emojis')} AS emoji
    WHERE emoji != ''
    """,
    "(emoji, tweet_id)",
)


Created `exploitation_analytics`.`bridge_tweet_hashtag`            192,677 rows
Created `exploitation_analytics`.`bridge_tweet_emoji`               12,666 rows


## 7. Denormalized Analytical Marts

Marts are query-friendly tables designed for dashboards, reports, and direct analysis. They intentionally denormalize dimensions and facts so users do not need to write many joins.

| Table | Grain | Purpose |
|---|---|---|
| `mart_climate_country_year` | One country per year | Dashboard-ready climate mart with derived metrics such as CO2 per capita, GDP per capita, and emission intensity. |
| `mart_temperature_trends` | One area per month per year | Time-series mart with annual average temperature change and rolling five-year monthly averages. |
| `mart_vehicle_emission_summary` | One make/model/class/fuel group | Summarizes average CO2 emissions and fuel consumption, with ranking inside each vehicle class. |
| `mart_disaster_tweet_features` | One row per tweet | ML/BI-ready tweet feature table without raw modelling joins. |
| `mart_disaster_type_profile` | One row per disaster type | Aggregated profile comparing tweet volume, text length, hashtag usage, alert-keyword share, and simple sentiment by disaster category. |

In [8]:
# Create dashboard-ready marts for climate, temperature, and vehicle-emission analysis.
# These marts join dimension labels back onto facts and add derived analytical metrics.
recreate_table(
    "mart_climate_country_year",
    f"""
    SELECT
        c.country_name AS country,
        y.year,
        y.decade,
        f.temperature_anomaly,
        f.average_temperature,
        f.co2_emissions,
        f.population,
        if(f.population = 0, NULL, f.co2_emissions / f.population) AS co2_per_capita,
        f.gdp,
        if(f.population = 0, NULL, f.gdp / f.population) AS gdp_per_capita,
        if(f.gdp = 0, NULL, f.co2_emissions / f.gdp) AS emission_intensity,
        f.renewable_energy_usage,
        f.forest_area,
        f.deforestation_rate,
        f.extreme_weather_events,
        f.policy_score,
        f.air_pollution_index,
        f.biodiversity_index,
        f.fossil_fuel_usage
    FROM {exploitation_ref('fact_climate_country_year')} f
    INNER JOIN {exploitation_ref('dim_country')} c USING country_key
    INNER JOIN {exploitation_ref('dim_year')} y USING year_key
    """,
    "(country, year)",
)

# Create tweet-focused marts: one feature-level table for ML and one aggregated profile table for BI.
recreate_table(
    "mart_temperature_trends",
    f"""
    SELECT
        a.area_name AS area,
        y.year,
        m.month_name AS month,
        f.temperature_change_value,
        avg(f.temperature_change_value) OVER (PARTITION BY a.area_name, y.year) AS annual_avg_temperature_change,
        avg(f.temperature_change_value) OVER (
            PARTITION BY a.area_name, m.month_name
            ORDER BY y.year
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ) AS rolling_5y_month_avg
    FROM {exploitation_ref('fact_temperature_area_month')} f
    INNER JOIN {exploitation_ref('dim_area')} a USING area_key
    INNER JOIN {exploitation_ref('dim_year')} y USING year_key
    INNER JOIN {exploitation_ref('dim_month')} m USING month_key
    """,
    "(area, year, month)",
)

recreate_table(
    "mart_vehicle_emission_summary",
    f"""
    SELECT
        v.make,
        v.model,
        v.vehicle_class,
        v.fuel_type,
        count() AS vehicle_record_count,
        avg(f.co2_emissions_g_km) AS avg_co2_emissions_g_km,
        avg(f.fuel_consumption_comb_l_100_km) AS avg_fuel_consumption_comb_l_100_km,
        avg(f.fuel_consumption_comb_mpg) AS avg_fuel_consumption_comb_mpg,
        rank() OVER (PARTITION BY v.vehicle_class ORDER BY avg(f.co2_emissions_g_km) DESC) AS emission_rank_in_class
    FROM {exploitation_ref('fact_vehicle_emission')} f
    INNER JOIN {exploitation_ref('dim_vehicle')} v USING vehicle_key
    GROUP BY v.make, v.model, v.vehicle_class, v.fuel_type
    """,
    "(vehicle_class, emission_rank_in_class, make, model)",
)


Created `exploitation_analytics`.`mart_climate_country_year`       100,000 rows
Created `exploitation_analytics`.`mart_temperature_trends`         241,893 rows
Created `exploitation_analytics`.`mart_vehicle_emission_summary`      1,856 rows


In [9]:
# Create tweet-focused marts: one feature-level table for ML and one aggregated profile table for BI.
recreate_table(
    "mart_disaster_tweet_features",
    f"""
    SELECT
        tweet_id,
        disaster_type,
        emoji_count,
        text_length,
        word_count,
        hashtag_count,
        mention_count,
        url_count,
        exclamation_count,
        question_count,
        contains_youtube,
        contains_alert_keyword,
        simple_sentiment_score,
        simple_sentiment_label
    FROM {exploitation_ref('fact_tweet_features')}
    """,
    "tweet_id",
)


recreate_table(
    "mart_disaster_type_profile",
    f"""
    SELECT
        disaster_type,
        count() AS tweet_count,
        avg(text_length) AS avg_text_length,
        avg(word_count) AS avg_word_count,
        avg(hashtag_count) AS avg_hashtag_count,
        avg(mention_count) AS avg_mention_count,
        avg(url_count) AS avg_url_count,
        avg(emoji_count) AS avg_emoji_count,
        avg(contains_alert_keyword) AS alert_keyword_share,
        avg(simple_sentiment_score) AS avg_simple_sentiment_score,
        countIf(simple_sentiment_label = 'positive') AS positive_tweets,
        countIf(simple_sentiment_label = 'negative') AS negative_tweets,
        countIf(simple_sentiment_label = 'neutral') AS neutral_tweets
    FROM {exploitation_ref('fact_tweet_features')}
    GROUP BY disaster_type
    """,
    "disaster_type",
)


Created `exploitation_analytics`.`mart_disaster_tweet_features`    127,527 rows
Created `exploitation_analytics`.`mart_disaster_type_profile`            5 rows


## 8. Validation and Preview

In [10]:
# List every exploitation table and validate that each output contains the expected number of rows.
exploitation_tables = client.query(
    f"""
    SELECT name
    FROM system.tables
    WHERE database = '{EXPLOITATION_DB}'
    ORDER BY name
    """
).result_rows

for (table_name,) in exploitation_tables:
    row_count = client.query(f"SELECT count() FROM {exploitation_ref(table_name)}").first_row[0]
    print(f"{table_name:<40} {row_count:>10,} rows")


bridge_tweet_emoji                           12,666 rows
bridge_tweet_hashtag                        192,677 rows
dim_area                                        247 rows
dim_country                                     195 rows
dim_disaster_type                                 5 rows
dim_month                                        17 rows
dim_vehicle                                   3,097 rows
dim_year                                        124 rows
fact_climate_country_year                   100,000 rows
fact_temperature_area_month                 241,893 rows
fact_tweet_features                         127,527 rows
fact_vehicle_emission                         5,988 rows
mart_climate_country_year                   100,000 rows
mart_disaster_tweet_features                127,527 rows
mart_disaster_type_profile                        5 rows
mart_temperature_trends                     241,893 rows
mart_vehicle_emission_summary                 1,856 rows


In [11]:
# Preview the main marts so the notebook output can be inspected immediately after execution.
preview_queries = {
    "mart_climate_country_year": f"SELECT * FROM {exploitation_ref('mart_climate_country_year')} LIMIT 5",
    "mart_temperature_trends": f"SELECT * FROM {exploitation_ref('mart_temperature_trends')} LIMIT 5",
    "mart_vehicle_emission_summary": f"SELECT * FROM {exploitation_ref('mart_vehicle_emission_summary')} LIMIT 5",
    "mart_disaster_type_profile": f"SELECT * FROM {exploitation_ref('mart_disaster_type_profile')} ORDER BY tweet_count DESC LIMIT 10",
}

for title, query in preview_queries.items():
    print(f"\n{title}")
    display(client.query_df(query))



mart_climate_country_year


,country,year,decade,temperature_anomaly,average_temperature,co2_emissions,population,co2_per_capita,gdp,gdp_per_capita,emission_intensity,renewable_energy_usage,forest_area,deforestation_rate,extreme_weather_events,policy_score,air_pollution_index,biodiversity_index,fossil_fuel_usage
0,country_1,1900,1900,-1.458531,-9.319760,5.735942e+05,1.961684e+08,0.002924,9.190033e+12,46847.676235,6.241481e-08,95.622335,0.548149,0.774289,41,43.001773,61.708875,55.262046,95.672073
1,country_1,1900,1900,-1.522098,16.997096,6.083792e+08,2.020119e+08,3.011601,3.694880e+12,18290.409641,1.646547e-04,17.645032,30.783018,4.958358,41,58.360755,147.254164,13.971719,11.643323
2,country_1,1900,1900,0.868484,1.096845,7.737906e+08,5.443298e+08,1.421547,1.941820e+12,3567.359459,3.984873e-04,80.802182,65.694293,4.710046,15,10.122462,283.977691,34.448477,44.694559
3,country_1,1900,1900,0.772039,13.235857,2.111143e+08,5.576762e+08,0.378561,3.466274e+12,6215.566881,6.090527e-05,46.673056,14.401780,2.034420,10,60.284283,150.584727,79.475557,57.699551
4,country_1,1901,1900,-0.703154,32.393594,6.998442e+08,1.494792e+09,0.468188,9.044301e+12,6050.543023,7.737958e-05,46.218131,54.731054,0.648111,30,17.905168,173.091676,9.140253,56.227115



mart_temperature_trends


,area,year,month,temperature_change_value,annual_avg_temperature_change,rolling_5y_month_avg
0,afghanistan,1961,april,-1.786,-0.020294,-1.786
1,afghanistan,1961,august,0.361,-0.020294,0.361
2,afghanistan,1961,dec-jan-feb,-0.763,-0.020294,-0.763
3,afghanistan,1961,december,0.546,-0.020294,0.546
4,afghanistan,1961,february,-1.787,-0.020294,-1.787



mart_vehicle_emission_summary


,make,model,vehicle_class,fuel_type,vehicle_record_count,avg_co2_emissions_g_km,avg_fuel_consumption_comb_l_100_km,avg_fuel_consumption_comb_mpg,emission_rank_in_class
0,rolls-royce,phantom drophead coupe,compact,z,4,398.500000,17.150000,16.5,1
1,rolls-royce,phantom coupe,compact,z,4,398.000000,17.150000,16.5,2
2,rolls-royce,dawn,compact,z,3,395.666667,16.933333,17.0,3
3,bentley,continental gt convertible,compact,z,1,389.000000,16.600000,17.0,4
4,bentley,continental supersports,compact,z,1,389.000000,16.600000,17.0,4



mart_disaster_type_profile


,disaster_type,tweet_count,avg_text_length,avg_word_count,avg_hashtag_count,avg_mention_count,avg_url_count,avg_emoji_count,alert_keyword_share,avg_simple_sentiment_score,positive_tweets,negative_tweets,neutral_tweets
0,hurricane,52084,123.279875,19.422049,0.984237,0.550764,0.073305,0.039436,0.080831,0.051206,9013,6750,36321
1,earthquake,38979,145.650555,20.441161,2.477437,0.403833,0.610842,0.260114,0.029785,0.154724,8016,2391,28572
2,flood,19013,153.954189,23.026719,1.378215,0.756535,0.259033,0.007521,0.032451,0.205544,4482,974,13557
3,wildfire,12996,144.917128,22.826716,0.870345,0.620037,0.087104,0.018082,0.055863,-0.006387,1968,2020,9008
4,cyclone,4455,186.810325,28.446914,1.645567,0.969921,0.089113,0.021324,0.053872,0.096745,900,468,3087
